In [ ]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [ ]:
from synth_extract.agents.llm import LLMBackend  # noqa: E402


from pydantic import BaseModel, ConfigDict, Field, field_validator
from synth_extract.agents.classification.schemas import CompletionMetadata

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import asyncio

In [ ]:
class PropertyExtractionResult(BaseModel):
    """Verbatim measured-property names extracted from a passage."""

    model_config = ConfigDict(extra="forbid", strict=True)

    properties: list[str] = Field(
        description=(
            "Property-name spans for which a corresponding property "
            "value is explicitly reported in the passage. Each string must be "
            "copied exactly from the passage. Do not include values, units, "
            "measurement methods, conditions, materials, or inferred properties."
        )
    )

    metadata: CompletionMetadata

    @field_validator("properties")
    @classmethod
    def validate_properties(cls, properties: list[str]) -> list[str]:
        """Require non-empty spans after model generation."""
        if any(not property_name.strip() for property_name in properties):
            raise ValueError("Property spans must be non-empty strings.")
        return properties

In [ ]:
import json
from pathlib import Path
from typing import Any


class PropertyDiscoveryLLM:
    """Extract measured-property spans from a paper using an LLMBackend."""

    def __init__(
        self,
        backend: LLMBackend,
        system_prompt_path: str | Path | None = None,
        user_prompt_path: str | Path | None = None,
    ) -> None:
        self.backend = backend
        if system_prompt_path is None or user_prompt_path is None:
            prompt_dir = self._find_prompt_dir()
        self.system_prompt_path = (
            Path(system_prompt_path)
            if system_prompt_path is not None
            else prompt_dir / "property_discovery.md"
        )
        self.user_prompt_path = (
            Path(user_prompt_path)
            if user_prompt_path is not None
            else prompt_dir / "user_prompt.md"
        )
        self.reload_prompts()

    @staticmethod
    def _find_prompt_dir() -> Path:
        """Locate prompts when Jupyter starts here or at the project root."""
        candidates = (Path.cwd(), Path.cwd() / "discovery")
        for candidate in candidates:
            if (candidate / "property_discovery.md").is_file() and (
                candidate / "user_prompt.md"
            ).is_file():
                return candidate
        raise FileNotFoundError(
            "Could not find property_discovery.md and user_prompt.md. "
            "Pass their paths explicitly."
        )

    def reload_prompts(self) -> None:
        """Reload both UTF-8 prompt files from disk."""
        self._system_prompt = self.system_prompt_path.read_text(
            encoding="utf-8"
        ).strip()
        self._user_prompt = self.user_prompt_path.read_text(
            encoding="utf-8"
        ).strip()

    @staticmethod
    def response_format() -> dict[str, Any]:
        """Require the bare string array specified by the system prompt."""
        return {
            "type": "json_schema",
            "json_schema": {
                "name": "measured_property_spans",
                "strict": True,
                "schema": {
                    "type": "array",
                    "items": {"type": "string"},
                },
            },
        }

    def build_messages(self, fulltext: str) -> list[dict[str, str]]:
        """Build the messages for one paper without calling the model."""
        if not isinstance(fulltext, str) or not fulltext.strip():
            raise ValueError("fulltext must be a non-empty string")
        return [
            {"role": "system", "content": self._system_prompt},
            {
                "role": "user",
                "content": self._user_prompt.format(
                    fulltext=fulltext.strip()
                ),
            },
        ]

    def render_request(self, fulltext: str) -> str:
        """Render the exact request body without sending it."""
        return self.backend.render_request(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    @staticmethod
    def _metadata(completion: Any) -> CompletionMetadata:
        choice = completion.choices[0]
        message = choice.message
        usage = getattr(completion, "usage", None)
        token_usage = None
        if usage is not None:
            token_usage = {
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(
                    usage, "completion_tokens", None
                ),
                "total_tokens": getattr(usage, "total_tokens", None),
            }
        return CompletionMetadata(
            model=getattr(completion, "model", None),
            created=getattr(completion, "created", None),
            finish_reason=getattr(choice, "finish_reason", None),
            stop_reason=getattr(choice, "stop_reason", None),
            reasoning=getattr(message, "reasoning", None),
            usage=token_usage,
        )

    @classmethod
    def _parse_completion(cls, completion: Any) -> PropertyExtractionResult:
        if not completion.choices:
            raise ValueError("The provider returned no completion choices.")

        choice = completion.choices[0]
        if choice.finish_reason == "length":
            raise ValueError("The property response reached the token limit.")

        refusal = getattr(choice.message, "refusal", None)
        if refusal:
            raise ValueError(f"The provider refused the request: {refusal}")

        content = choice.message.content
        if not content:
            raise ValueError("The provider returned an empty response.")

        properties = json.loads(content)
        if not isinstance(properties, list):
            raise ValueError("The response must be a JSON array.")
        if not all(isinstance(item, str) for item in properties):
            raise ValueError("Every property span must be a string.")
        return PropertyExtractionResult(
            properties=properties,
            metadata=cls._metadata(completion),
        )

    def extract_raw(self, fulltext: str) -> Any:
        """Synchronously return the backend's raw completion."""
        return self.backend.create_completion(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    def extract(self, fulltext: str) -> PropertyExtractionResult:
        """Synchronously extract and validate properties from a paper."""
        return self._parse_completion(self.extract_raw(fulltext))

    async def aextract_raw(self, fulltext: str) -> Any:
        """Asynchronously return the backend's raw completion."""
        return await self.backend.acreate_completion(
            messages=self.build_messages(fulltext),
            response_format=self.response_format(),
        )

    async def aextract(self, fulltext: str) -> PropertyExtractionResult:
        """Asynchronously extract and validate properties from a paper."""
        completion = await self.aextract_raw(fulltext)
        return self._parse_completion(completion)

    def health_check(self) -> bool:
        """Return ``True`` when the endpoint responds to a model-list request."""
        self.backend.list_models()
        return True

In [ ]:
host="localhost"
port="8000"
base_url=f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens=16384
extra_body = {"chat_template_kwargs":{"enable_thinking":False}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=300,
    max_tokens=max_tokens,
    extra_body=extra_body,
)

In [ ]:
base_path = Path(".")
system_prompt_path = base_path / "property_discovery.md"
user_prompt_path = base_path / "user_prompt.md"

In [ ]:
discoveryLM = PropertyDiscoveryLLM(backend=backend,
                                   system_prompt_path=system_prompt_path,
                                   user_prompt_path=user_prompt_path)

In [ ]:
discoveryLM.health_check()

In [ ]:
full_text_path = Path("/Users/kevinge/Work/Data Extraction/synth_extract/data/fulltext")

In [ ]:
discoveryLM.backend.extra_body = {"chat_template_kwargs":{"enable_thinking":False}}
discoveryLM.backend.config()

In [ ]:
# Pick a paper
uid = "ID000192337"
source = "elsevier"

# Construct full-text path
fulltext_path = (
    Path(full_text_path)
    / source
    / uid
    / f"{uid}.md"
)

print("Source:", source)
print("Full text:", fulltext_path)

if not fulltext_path.exists():
    raise FileNotFoundError(fulltext_path)

# Load Markdown
full_text = fulltext_path.read_text(encoding="utf-8")

print(f"Characters: {len(full_text):,}")

# Classify
result = discoveryLM.extract(full_text)

result

In [ ]:
print(result.metadata.reasoning)

In [ ]:
# Run property discovery concurrently over all Markdown full-text files.
import asyncio
from collections import Counter
import json
import random
from pathlib import Path

fulltext_root = Path(full_text_path).resolve()
all_fulltext_paths: list[Path] = sorted(fulltext_root.rglob("*.md"))
if not all_fulltext_paths:
    raise FileNotFoundError(f"No Markdown files found below {fulltext_root}")

# Set limit=None to process every path. A positive integer selects that many
# paths randomly. Set random_seed to an integer for a reproducible sample.
limit: int | None = None
random_seed: int | None = None
max_parallel_requests = 8


async def discover_file_async(
    number: int,
    total: int,
    fulltext_path: Path,
    semaphore: asyncio.Semaphore,
) -> tuple[str, list[str] | None, str | None]:
    """Discover properties for one file without exceeding the API limit."""
    uid = fulltext_path.parent.name
    source = fulltext_path.parent.parent.name

    try:
        async with semaphore:
            full_text = await asyncio.to_thread(
                fulltext_path.read_text,
                encoding="utf-8",
            )
            result = await discoveryLM.aextract(full_text)

        properties = result.properties
        usage = result.metadata.usage
        token_summary = (
            f"input={usage.prompt_tokens}, output={usage.completion_tokens}, "
            f"total={usage.total_tokens}"
            if usage is not None
            else "unavailable"
        )
        print(
            f"[{number}/{total}] Source: {source} | UID: {uid} | "
            f"Properties: {len(properties)} | Tokens: {token_summary}"
        )
        return uid, properties, None

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        print(
            f"[{number}/{total}] Source: {source} | UID: {uid} | "
            f"Error: {error}"
        )
        return uid, None, error


async def discover_properties_async(
    paths: list[Path],
    *,
    limit: int | None = None,
    max_parallel_requests: int = 8,
    random_seed: int | None = None,
) -> tuple[list[Path], dict[str, list[str]], dict[str, str]]:
    """Select paths, run discovery, and return paths/results/errors."""
    if max_parallel_requests < 1:
        raise ValueError("max_parallel_requests must be at least 1")
    if limit is not None:
        if limit < 1:
            raise ValueError("limit must be a positive integer or None")
        if limit > len(paths):
            raise ValueError(
                f"limit={limit} exceeds the {len(paths)} available files"
            )
        selected_paths = random.Random(random_seed).sample(paths, k=limit)
    else:
        selected_paths = list(paths)

    selected_uids = [path.parent.name for path in selected_paths]
    duplicate_uids = sorted(
        uid for uid, count in Counter(selected_uids).items() if count > 1
    )
    if duplicate_uids:
        raise ValueError(
            f"UIDs must be unique dictionary keys; duplicates: {duplicate_uids}"
        )

    semaphore = asyncio.Semaphore(max_parallel_requests)
    tasks = [
        asyncio.create_task(
            discover_file_async(
                number,
                len(selected_paths),
                path,
                semaphore,
            )
        )
        for number, path in enumerate(selected_paths, start=1)
    ]
    outcomes = await asyncio.gather(*tasks)

    results_by_uid = {
        uid: properties
        for uid, properties, error in outcomes
        if properties is not None and error is None
    }
    errors_by_uid = {
        uid: error
        for uid, properties, error in outcomes
        if error is not None
    }
    return selected_paths, results_by_uid, errors_by_uid


(
    selected_fulltext_paths,
    property_json_by_uid,
    property_discovery_errors,
) = await discover_properties_async(
    all_fulltext_paths,
    limit=limit,
    max_parallel_requests=max_parallel_requests,
    random_seed=random_seed,
)

property_discovery_json = json.dumps(
    property_json_by_uid,
    indent=2,
    ensure_ascii=False,
)
print(
    f"Completed: {len(property_json_by_uid)} | "
    f"Failed: {len(property_discovery_errors)}"
)

In [ ]:
# Vocabulary saturation / species accumulation curves.
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator

minimum_paper_count = 3
if not property_json_by_uid:
    raise ValueError("property_json_by_uid is empty; run discovery first.")

paper_numbers = [0]
cumulative_unique_properties = [0]
cumulative_properties_in_at_least_two_papers = [0]
cumulative_properties_in_at_least_three_papers = [0]
seen_properties: set[str] = set()
property_paper_counts: Counter[str] = Counter()

for paper_number, properties in enumerate(
    property_json_by_uid.values(),
    start=1,
):
    # A repeated span within one paper counts only once toward prevalence.
    properties_in_paper = set(properties)
    seen_properties.update(properties_in_paper)
    property_paper_counts.update(properties_in_paper)

    paper_numbers.append(paper_number)
    cumulative_unique_properties.append(len(seen_properties))
    cumulative_properties_in_at_least_two_papers.append(
            sum(
                paper_count >= 2
                for paper_count in property_paper_counts.values()
            )
        )
    cumulative_properties_in_at_least_three_papers.append(
        sum(
            paper_count >= minimum_paper_count
            for paper_count in property_paper_counts.values()
        )
    )

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    paper_numbers,
    cumulative_unique_properties,
    linewidth=2,
    label="All unique properties",
)
ax.plot(
    paper_numbers,
    cumulative_properties_in_at_least_two_papers,
    linewidth=2,
    label=f"Properties present in at least {2} papers",
)
ax.plot(
    paper_numbers,
    cumulative_properties_in_at_least_three_papers,
    linewidth=2,
    label=f"Properties present in at least {minimum_paper_count} papers",
)

ax.set_xlabel("Number of papers processed")
ax.set_ylabel("Cumulative number of unique properties")
ax.set_title("Property vocabulary saturation")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_yscale("log")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

property_saturation_curve = pd.DataFrame(
    {
        "papers_processed": paper_numbers,
        "cumulative_unique_properties": cumulative_unique_properties,
        "cumulative_properties_in_at_least_3_papers": (
            cumulative_properties_in_at_least_three_papers
        ),
    }
)

In [ ]:
property_paper_counts